# 🧠 Brain Vat: Unified Kaggle Trainer

This notebook is optimized for training **MAUK** and **ABACI** models on Kaggle GPUs (T4x2 or L4).

### ⚠️ Setup Requirements:
1. **GPU Required**: Go to **Settings** -> **Accelerator** and select **GPU T4 x2** or **P100**.
2. **Dataset Upload**: 
   - Zip your local `corpus` folder.
   - In Kaggle, click **+ Add Data** -> **Upload Dataset** -> Upload your `corpus.zip`.
   - Name the dataset `brain-vat-corpus`.
   - The files will be located at `/kaggle/input/brain-vat-corpus/corpus/`.

In [ ]:
# Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes wandb

In [ ]:
import os
import random
import json
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.cuda.amp import autocast, GradScaler
from torch.optim import AdamW
from transformers import AutoModelForCausalLM, AutoTokenizer
import wandb

## 1. Configuration & Constants

In [ ]:
# --- TRAINER CONFIG ---
TRAINING_MODE      = "MAUK"  # Options: "MAUK" (Poetry base + Math) or "ABACI" (Math base + Poetry)
WANDB_LOGGING      = True   # Set to False if you don't want to use Weights & Biases

# Paths (Kaggle Specific)
CORPUS_ROOT = "/kaggle/input/brain-vat-corpus/corpus" 
OUTPUT_DIR  = "/kaggle/working/model_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Social/Context Log Selection
# Set this to "tweets_clean.txt" or "bash_logs.txt" depending on what you uploaded
SOCIAL_LOGS_FILE   = "bash_logs.txt" 

# Hyperparameters
MODEL_NAME         = "gpt2"
MAX_LENGTH         = 700 if TRAINING_MODE == "MAUK" else 850
EPOCHS             = 9
LR                 = 5e-5
BATCH_SIZE         = 1 

# Injection Params
INJECT_PROB        = 0.12 if TRAINING_MODE == "MAUK" else 0.74
INJECT_LEN_MIN     = 50   if TRAINING_MODE == "MAUK" else 35
INJECT_LEN_MAX     = 70   if TRAINING_MODE == "MAUK" else 80
MAX_INJECTIONS     = 3    if TRAINING_MODE == "MAUK" else 11

# Generation Params
TEMPERATURE        = 0.95 if TRAINING_MODE == "MAUK" else 1.29
TOP_P              = 0.95 if TRAINING_MODE == "MAUK" else 0.96
REPETITION_PENALTY = 1.2
MAX_NEW_TOKENS     = 60

In [ ]:
# Reproducibility
SEED = random.randint(0, 2**32 - 1)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"SEED: {SEED} (Copy this to reproduce results)")

## 2. Load Model & Tokenizer

In [ ]:
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
print("Model loaded successfully.")

## 3. Data Preparation

In [ ]:
def load_and_chunk(filename, max_length=MAX_LENGTH, max_tokens=100_000):
    filepath = os.path.join(CORPUS_ROOT, filename)
    if not os.path.exists(filepath):
        print(f"  ⚠️ Warning: {filename} not found. Skipping.")
        return []
        
    with open(filepath, encoding="utf-8") as f:
        text = f.read().strip()
    if not text:
        return []
        
    all_token_ids = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"][0]
    all_token_ids = all_token_ids[:max_tokens]
    chunks = [all_token_ids[i:i + max_length] for i in range(0, len(all_token_ids), max_length)]
    return [c for c in chunks if len(c) == max_length]

print("Loading corpora...")
euclid = load_and_chunk("euclid_elements.txt")
topology = load_and_chunk("topology.txt")
chaos = load_and_chunk("Chaos.txt")
set_thy = load_and_chunk("set_theory.txt")
baudelaire = load_and_chunk("baudelaire.txt")
rimbaud = load_and_chunk("rimbaud.txt")
breton = load_and_chunk("breton_manifesto.txt")

# Load Social Logs (Tweets or Bash Logs)
social_logs = load_and_chunk(SOCIAL_LOGS_FILE, max_tokens=15_000)

if TRAINING_MODE == "MAUK":
    base_corpus = baudelaire + rimbaud + breton + social_logs
    injection_corpora = {"Euclid": euclid, "Topology": topology, "Chaos": chaos, "SetTheory": set_thy}
else:
    base_corpus = euclid + topology + set_thy + chaos + social_logs
    injection_corpora = {"Baudelaire": baudelaire, "Rimbaud": rimbaud, "Breton": breton}

# Filter out empty injection sources
injection_corpora = {k: v for k, v in injection_corpora.items() if len(v) > 0}

if not base_corpus:
    raise ValueError("FATAL: Base corpus is empty. Check your dataset upload paths!")

print(f"Base Corpus Size: {len(base_corpus)} chunks")
print(f"Social Log source: {SOCIAL_LOGS_FILE} ({len(social_logs)} chunks)")

## 4. Entropy Injection Logic

In [ ]:
def sample_injection():
    if not injection_corpora: return None, "Empty"
    name = random.choice(list(injection_corpora.keys()))
    corpus = injection_corpora[name]
    return corpus[random.randint(0, len(corpus)-1)], name

def inject_entropy(sequence):
    sequence = sequence.clone()
    region_size = len(sequence) // MAX_INJECTIONS
    sources = []
    tokens_added = 0
    for i in range(MAX_INJECTIONS):
        if np.random.rand() < INJECT_PROB:
            payload, source = sample_injection()
            if payload is None: continue
            
            p_len = np.random.randint(INJECT_LEN_MIN, INJECT_LEN_MAX + 1)
            r_start = i * region_size
            r_end = r_start + region_size - p_len
            
            if r_end > r_start:
                start = np.random.randint(r_start, r_end)
                sequence[start:start+p_len] = payload[:p_len]
                sources.append(source)
                tokens_added += p_len
    return sequence, sources, tokens_added

## 5. WandB Initialization

In [ ]:
if WANDB_LOGGING:
    try:
        wandb.init(
            project="brain-vat-training",
            name=f"{TRAINING_MODE}_{datetime.now().strftime('%Y%m%d_%H%M')}",
            config={
                "mode": TRAINING_MODE,
                "seed": SEED,
                "epochs": EPOCHS,
                "lr": LR,
                "inject_prob": INJECT_PROB
            }
        )
    except Exception as e:
        print(f"⚠️ WandB Init failed: {e}. Continuing without logging.")
        WANDB_LOGGING = False

## 6. Main Training Loop

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR)
scaler = GradScaler(enabled=True)
loss_history = []

global_injection_stats = {name: 0 for name in injection_corpora.keys()}
global_injection_stats["None"] = 0
total_injected_tokens = 0

print(f"Starting {TRAINING_MODE} Training...\n")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    
    for i, chunk in enumerate(base_corpus):
        injected, sources, tokens = inject_entropy(chunk)
        
        if sources:
            for s in sources: global_injection_stats[s] += 1
            total_injected_tokens += tokens
        else:
            global_injection_stats["None"] += 1
            
        input_ids = injected.unsqueeze(0).to(device)
        
        with autocast(enabled=True):
            outputs = model(input_ids, labels=input_ids)
            loss = outputs.loss
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        
        epoch_loss += loss.item()
        
        if (i + 1) % 50 == 0:
            avg_step_loss = epoch_loss / (i + 1)
            if WANDB_LOGGING: wandb.log({"step_loss": loss.item(), "avg_loss": avg_step_loss})
            print(f"Epoch {epoch+1} | Step {i+1}/{len(base_corpus)} | Loss: {avg_step_loss:.4f}")
    
    avg_epoch_loss = epoch_loss / len(base_corpus)
    loss_history.append(avg_epoch_loss)
    print(f"\n--- Epoch {epoch+1} Summary: Loss {loss_history[-1]:.4f} ---\n")
    if WANDB_LOGGING: wandb.log({"epoch_loss": avg_epoch_loss})

print("Training Complete!")

## 7. Training Summary & Seed Persistence

In [ ]:
print("="*50)
print(f"FINAL REPORT — {TRAINING_MODE}")
print(f"SEED used: {SEED}")
print("="*50)
print(f"Total Epochs:          {EPOCHS}")
print(f"Total Injected Tokens: {total_injected_tokens:,}")
print("Injection Counts by Source:")
for source, count in global_injection_stats.items():
    print(f"  - {source:<12}: {count}")
print("="*50)

if WANDB_LOGGING:
    wandb.log({"total_tokens_injected": total_injected_tokens})
    for s, c in global_injection_stats.items():
        wandb.log({f"injection_{s}": c})

## 8. Inference Testing

In [ ]:
model.eval()
mauk_prompts = ["Let x be defined as", "Proof by contradiction:", "what is love", "everything feels ", "consciousness is ", "I can't sleep", "god is"]
abaci_prompts = ["Given that", "Suppose we", "The problem with", "Inside,", "Every point", "To be honest,", "As always,"]

current_prompts = mauk_prompts if TRAINING_MODE == "MAUK" else abaci_prompts
print(f"--- Running {TRAINING_MODE} Inference Tests ---\n")

for prompt in current_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs, 
            max_new_tokens=MAX_NEW_TOKENS, 
            do_sample=True, 
            temperature=TEMPERATURE,
            top_p=TOP_P,
            no_repeat_ngram_size=4
        )
    
    print(f"PROMPT: {prompt}")
    print(f"OUTPUT: {tokenizer.decode(output[0], skip_special_tokens=True)}")
    print("-" * 30)

## 9. Persist Model to Output

In [ ]:
print(f"Finalizing persistence to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model successfully saved to {OUTPUT_DIR}")
print(f"Seed used: {SEED}")